# 02 — Feature Engineering

Build the three core joined tables and derive all features:
- **customer_base**: one row per account (static + behavioral + fraud flags)
- **customer_monthly**: one row per account per month (lag, rolling, ratio features)
- **transaction_base_enriched**: transaction-level fraud flags

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline, load_table_as_df
from src import preprocessing as pp
from src import features as feat

pd.set_option('display.max_columns', 50)

In [ ]:
con, counts = load_pipeline(verbose=True)

## 1. Build `customer_base` — One Row Per Account

In [ ]:
customer_base = feat.build_customer_base(con)
print(f"\ncustomer_base shape: {customer_base.shape}")
print(f"Unique accounts: {customer_base['current_account_nbr'].nunique()}")

In [ ]:
# Verify key derived features
print("\n=== Derived Feature Summary ===")
derived_cols = [
    'delinquent_cycle_count', 'max_delinquency_level', 'risk_flag_count',
    'avg_monthly_sales_6m', 'sales_trend_slope',
    'utilization_ratio', 'otb_ratio', 'has_fraud_case'
]
for col in derived_cols:
    if col in customer_base.columns:
        s = customer_base[col]
        print(f"  {col}: mean={s.mean():.3f}, median={s.median():.3f}, "
              f"min={s.min():.3f}, max={s.max():.3f}")

In [ ]:
# Fraud case flag distribution
fraud_counts = customer_base['has_fraud_case'].value_counts()
print(f"\nFraud case distribution:")
print(fraud_counts)
print(f"Fraud rate: {fraud_counts.get(1, 0) / len(customer_base) * 100:.2f}%")

## 2. Build `customer_monthly` — Account × Month

In [ ]:
customer_monthly = feat.build_customer_monthly(con)
print(f"\ncustomer_monthly shape: {customer_monthly.shape}")
print(f"Unique accounts: {customer_monthly['current_account_nbr'].nunique()}")
print(f"Date range: {customer_monthly['month'].min()} to {customer_monthly['month'].max()}")

In [ ]:
# Add lag features
customer_monthly = feat.add_lag_features(customer_monthly)
print("\nLag features added:")
lag_cols = ['spend_lag_1', 'spend_lag_2', 'spend_lag_3', 
            'spend_rolling_mean_3', 'spend_rolling_std_3']
for col in lag_cols:
    if col in customer_monthly.columns:
        non_null = customer_monthly[col].notna().sum()
        print(f"  {col}: {non_null:,} non-null values")

In [ ]:
# Add spend-to-limit ratio
customer_monthly = feat.add_spend_to_limit_ratio(customer_monthly, customer_base)
print(f"\nspend_to_limit_ratio: mean={customer_monthly['spend_to_limit_ratio'].mean():.4f}")

In [ ]:
# Monthly spend distribution
monthly_agg = customer_monthly.groupby('month').agg(
    total_spend=('total_spend', 'sum'),
    avg_spend=('total_spend', 'mean'),
    active_accounts=('current_account_nbr', 'nunique')
).reset_index()

fig = px.line(monthly_agg, x='month', y='total_spend',
              title='Total Monthly Spend Across All Accounts',
              labels={'total_spend': 'Total Spend ($)', 'month': 'Month'})
fig.update_traces(line=dict(width=3, color='#e74c3c'))
fig.update_layout(template='plotly_white', height=400)
fig.write_image("../outputs/figures/feature_monthly_spend.png", scale=2)
fig.show()

## 3. Build `transaction_base_enriched` — Fraud-Flagged Transactions

In [ ]:
txn_enriched = feat.build_transaction_enriched(con)
print(f"\ntransaction_base_enriched shape: {txn_enriched.shape}")
print(f"Flagged fraud transactions: {txn_enriched['is_fraud_txn'].sum()}")
print(f"Fraud transaction rate: {txn_enriched['is_fraud_txn'].mean()*100:.4f}%")

## 4. Build Q4 Feature Matrix

In [ ]:
train_df, test_df, feature_cols, target_col = feat.build_q4_feature_matrix(
    customer_monthly, customer_base, cutoff_month='2024-10-01'
)

print(f"\nQ4 Feature Matrix:")
print(f"  Train: {train_df.shape} (months before Oct 2024)")
print(f"  Test:  {test_df.shape} (Oct-Dec 2024)")
print(f"  Features: {len(feature_cols)}")
print(f"  Feature names: {feature_cols}")

In [ ]:
# Feature value ranges
print("\nFeature Statistics (Training Set):")
print(train_df[feature_cols].describe().round(2).to_string())

## 5. Build Anomaly Detection Features

In [ ]:
anomaly_features = feat.build_anomaly_features(con)
print(f"\nAnomaly features shape: {anomaly_features.shape}")
print(anomaly_features.describe().round(2))

## 6. Save Intermediate DataFrames

In [ ]:
# Save for downstream notebooks
customer_base.to_parquet('../outputs/predictions/customer_base.parquet', index=False)
customer_monthly.to_parquet('../outputs/predictions/customer_monthly.parquet', index=False)
anomaly_features.to_parquet('../outputs/predictions/anomaly_features.parquet', index=False)

print("Saved:")
print("  - customer_base.parquet")
print("  - customer_monthly.parquet")
print("  - anomaly_features.parquet")

## Summary

| Table | Shape | Key Features |
|-------|-------|-------------|
| `customer_base` | One row per account | Sales trend, utilization ratio, delinquency, fraud flags |
| `customer_monthly` | Account × month | Lag features, rolling stats, spend-to-limit ratio |
| `transaction_enriched` | Per transaction | `is_fraud_txn` flag from fraud_claim_tran join |
| Q4 feature matrix | Train/Test split | Temporal: Jan-Sep train, Oct-Dec test |